# Virtual warehouses

Provides CPU, memory, and temporary storage, to perform the following operations in a Snowflake session:

- Executing SQL SELECT statements that require compute resources
- Performing DML operations, such as:
- Updating rows in tables (DELETE , INSERT , UPDATE).
- Loading data into tables (COPY INTO <table>).
- Unloading data from tables (COPY INTO <location>).


        CREATE [ OR REPLACE ] WAREHOUSE [ IF NOT EXISTS ] <name>
               [ [ WITH ] objectProperties ]
               [ [ WITH ] TAG ( <tag_name> = '<tag_value>' [ , <tag_name> = '<tag_value>' , ... ] ) ]
               [ objectParams ]


        objectProperties ::=
          WAREHOUSE_TYPE = { STANDARD | 'SNOWPARK-OPTIMIZED' }
          WAREHOUSE_SIZE = { XSMALL | SMALL | MEDIUM | LARGE | XLARGE | XXLARGE | XXXLARGE | X4LARGE | X5LARGE | X6LARGE }
          GENERATION = { '1' | '2' }
          RESOURCE_CONSTRAINT = { STANDARD_GEN_1 | STANDARD_GEN_2 | MEMORY_1X | MEMORY_1X_x86 | MEMORY_16X | MEMORY_16X_x86 | MEMORY_64X | MEMORY_64X_x86 }
          MAX_CLUSTER_COUNT = <num>
          MIN_CLUSTER_COUNT = <num>
          SCALING_POLICY = { STANDARD | ECONOMY }
          AUTO_SUSPEND = { <num> | NULL }
          AUTO_RESUME = { TRUE | FALSE }
          INITIALLY_SUSPENDED = { TRUE | FALSE }
          RESOURCE_MONITOR = <monitor_name>
          COMMENT = '<string_literal>'
          ENABLE_QUERY_ACCELERATION = { TRUE | FALSE }
          QUERY_ACCELERATION_MAX_SCALE_FACTOR = <num>

          objectParams ::=
          MAX_CONCURRENCY_LEVEL = <num>
          STATEMENT_QUEUED_TIMEOUT_IN_SECONDS = <num>
          STATEMENT_TIMEOUT_IN_SECONDS = <num>


## WAREHOUSE_TYPE 

    DEFAULT = STANDARD

### STANDARD
Normal warehouses.
X-Small to 6X-Large

### Snowpark-optimized
Recommended for workloads that have large memory requirements such as ML training use cases.

## WAREHOUSE_SIZE 

    DEFAULT = XSMALL
    DEFAULT (SNOWPARK) = MEDIUM


| Size    | Standard Gen1 Credits/Hour | Standard Gen2 AWS/GCP (+35%) | Standard Gen2 Azure (+25%) | Snowpark‑Optimized Credits/Hour |
|---------|-----------------------------|-------------------------------|-----------------------------|----------------------------------|
| XS      | 1                           | 1.35                          | 1.25                        | N/A                              |
| S       | 2                           | 2.70                          | 2.50                        | N/A                              |
| M       | 4                           | 5.40                          | 5.00                        | 6                                |
| L       | 8                           | 10.80                         | 10.00                       | 12                               |
| XL      | 16                          | 21.60                         | 20.00                       | 24                               |
| 2XL     | 32                          | 43.20                         | 40.00                       | 48                               |
| 3XL     | 64                          | 86.40                         | 80.00                       | 96                               |
| 4XL     | 128                         | 172.80                        | 160.00                      | 192                              |
| 5XL     | 256                         | N/A (Gen2 not supported)      | N/A (Gen2 not supported)    | 384                              |
| 6XL     | 512                         | N/A (Gen2 not supported)      | N/A (Gen2 not supported)    | 768                              |
    

## GENERATION 

    DEFAULT = '1'

Gen2 is an updated version of the current standard virtual warehouse in Snowflake, focused on improving performance for analytics and data engineering workloads.   

- GEN2 = gene
- GENERATION applies only to standard warehouses    
- Need to be defined as string "1"

## RESOURCE_CONSTRAINT 

    { STANDARD_GEN_1 | STANDARD_GEN_2 | MEMORY_1X| MEMORY_1X_x86 | MEMORY_16X | MEMORY_16X_x86 | MEMORY_64X | MEMORY_64X_x86 }

Specifies the memory and CPU architecture for Snowpark-optimized warehouses, or generation 1 or generation 2 capabilities for standard warehouses.

Controls:

- memory per node
- CPU architecture (default ARM vs x86)
- generation of standard warehouse (Gen1 vs Gen2)

## MAX_CLUSTER_COUNT 

Specifies the maximum number of clusters for a multi-cluster warehouse. For a single-cluster warehouse, this value is always 1.

## MIN_CLUSTER_COUNT 

Specifies the minimum number of clusters for a multi-cluster warehouse (only applies to multi-cluster warehouses).

## MAXIMIZED VS AUTO-SCALE MODE

If MIN_CLUSTER_COUNT and MAX_CLUSTER_COUNT parameters are equal  the warehouse runs in Maximized mode.
If MIN_CLUSTER_COUNT is less than MAX_CLUSTER_COUNT, the warehouse runs in Auto-scale mode

### Maximized
- Snowflake starts all the clusters so that maximum resources are available while the warehouse is running

- Effective for statically controlling the available compute resources, particularly if you have large numbers of concurrent user sessions and/or queries and the numbers do not fluctuate significantly.

### Auto-scale 

Snowflake starts and stops clusters as needed to dynamically manage the load on the warehouse.

## SCALING_POLICY 

Policy for automatically starting and shutting down clusters in a multi-cluster warehouse running in Auto-scale mode

### STANDARD
Minimizes queuing by starting clusters.

#### Adding
When a query is queued, or if Snowflake estimates the currently running clusters don’t have enough resources to handle any additional queries, Snowflake increases the number of clusters in the warehouse.

#### Removing
Snowflake shuts down one or more of the least-loaded clusters when the queries running on them finish. When the cluster count is higher than 10, Snowflake might shut down multiple clusters at a time. When the cluster count is 10 or less, Snowflake shuts down the idle clusters one at a time


### ECONOMY
Conserves credits by favoring keeping running clusters fully-loaded

#### Adding
Only if the system estimates there’s enough query load to keep the cluster busy for at least 6 minutes.

#### Removing
Snowflake marks the least-loaded cluster for shutdown if it estimates the cluster has less than 6 minutes of work left to do.

## AUTO_SUSPEND 

        Default =  600 (10 minutes)

Specifies the number of seconds of inactivity after which a warehouse is automatically suspended.

## AUTO_RESUME 

        Default = TRUE

Specifies whether to automatically resume a warehouse when a SQL statement (for example, query) is submitted to it.

## INITIALLY_SUSPENDED

        Default = FALSE

Specifies whether the warehouse is created initially in the ‘Suspended’ state.

## RESOURCE_MONITOR 

Specifies the name of a resource monitor that is explicitly assigned to the warehouse

## ENABLE_QUERY_ACCELERATION 

Can accelerate parts of the query workload in a warehouse.

it can improve overall warehouse performance by reducing the impact of outlier queries, which are queries that use more resources than the typical query. The query acceleration service does this by offloading portions of the query processing work to shared compute resources that are provided by the service.

### SQL commands that QAS can accelerate

- SELECT
- INSERT
- CREATE TABLE AS SELECT (CTAS)
- COPY INTO <table>

## QUERY_ACCELERATION_MAX_SCALE_FACTOR : 0 - 100

        Default = 8

Specifies the maximum scale factor for leasing compute resources for query acceleration. The scale factor is used as a multiplier based on warehouse size.

Setting the QUERY_ACCELERATION_MAX_SCALE_FACTOR to 0 eliminates the limit and allows queries to lease as many resources as necessary and as available to service the query.

Regardless of the QUERY_ACCELERATION_MAX_SCALE_FACTOR value, the amount of available compute resources for query acceleration is bound by the available resources in the service and the number of other concurrent requests. For more details, refer to Adjusting the scale factor.


## Optional parametrs

### MAX_CONCURRENCY_LEVEL

    Default = 8

Specifies the concurrency level for SQL statements.

### STATEMENT_QUEUED_TIMEOUT_IN_SECONDS 

    Default = 0

Time, in seconds, a SQL statement (query, DDL, DML, etc.) can be queued on a warehouse before it is canceled by the system.

### STATEMENT_TIMEOUT_IN_SECONDS 

    Default = 172800 (2 days)

time, in seconds, after which a running SQL statement (query, DDL, DML, etc.) is canceled by the system.

- Max 7 days


##Types:

### Standard



## Credits vs Size


## Data Loading

Increasing the size of a warehouse does not always improve data loading performance. Data loading performance is influenced more by the number of files being loaded (and the size of each file) than the size of the warehouse.


## Impact on query processing

impact the amount of time required to execute queries submitted to the warehouse, particularly for larger, more complex queries